# Team 7 · Colab launcher

Runs the repository scripts on a Colab GPU. All model code lives in `src/`; this notebook only calls it.

1. Runtime → Change runtime type → **T4 GPU**.
2. Run sections 1–3 once.
3. In section 4, run **only the model assigned to you**, so the team trains several models at once on different accounts.
4. Run section 5 to save your results.

## 1. Setup

In [ ]:
!git clone -q https://github.com/r4fo/team7-argument-segmentation.git
%cd /content/team7-argument-segmentation
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/team7_checkpoints

## 2. Unit tests (seconds)

In [ ]:
!python -m pytest -q tests/

## 3. Smoke test (about 5 minutes)

Each script on a tiny sample, 1 epoch. The scores mean nothing; this only proves every model runs end to end before the long runs. Every line should finish with a results line and no error.

In [ ]:
!python src/baselines/crf_features.py --smoke
!python src/baselines/bilstm_crf.py --smoke --out-dir /content/drive/MyDrive/team7_checkpoints
!python src/trainers/train_token_clf.py --model bert-base-cased --decoder softmax --smoke --out-dir /content/drive/MyDrive/team7_checkpoints
!python src/trainers/train_token_clf.py --model bert-base-cased --decoder crf --smoke --out-dir /content/drive/MyDrive/team7_checkpoints
!python src/trainers/train_token_clf.py --model microsoft/deberta-v3-base --decoder crf --smoke --out-dir /content/drive/MyDrive/team7_checkpoints

## 4. Full runs: pick your model

Times are rough estimates for a T4. Checkpoints go to Google Drive. If the session disconnects, rerun setup and the same cell with `--resume` added (transformers only).

**Row 2 · CRF with hand features** (CPU, about 30–60 min; any runtime type works)

In [ ]:
!python src/baselines/crf_features.py

**Row 3 · BiLSTM-CRF** (about 30–60 min)

In [ ]:
!python src/baselines/bilstm_crf.py --out-dir /content/drive/MyDrive/team7_checkpoints

**Row 4 · BERT + softmax** (about 1.5–2 h)

In [ ]:
!python src/trainers/train_token_clf.py --model bert-base-cased --decoder softmax --out-dir /content/drive/MyDrive/team7_checkpoints

**Row 5 · BERT + CRF** (about 1.5–2 h). Then the truncation analysis on the same model (about 10 min).

In [ ]:
!python src/trainers/train_token_clf.py --model bert-base-cased --decoder crf --out-dir /content/drive/MyDrive/team7_checkpoints

In [ ]:
!python src/trainers/train_token_clf.py --model bert-base-cased --decoder crf --predict-only --truncate --out-dir /content/drive/MyDrive/team7_checkpoints

**Row 6 · DeBERTa-v3 + CRF** (about 3 h). If it runs out of GPU memory, add `--batch-size 4 --grad-accum 2`.

In [ ]:
!python src/trainers/train_token_clf.py --model microsoft/deberta-v3-base --decoder crf --out-dir /content/drive/MyDrive/team7_checkpoints

## 5. Save results

Zips the result files and the predictions (used for the error analysis) and downloads them. Send the zip in the chat.

In [ ]:
!mkdir -p /content/drive/MyDrive/team7_results
!cp results/*.json /content/drive/MyDrive/team7_results/
!cd results && zip -q -r /content/team7_results.zip *.json predictions
from google.colab import files
files.download('/content/team7_results.zip')